In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import tejapi
import matplotlib
import tqdm
import pandas as pd
import numpy as np
from catboost import CatBoostRanker, Pool
from catboost import CatBoostClassifier
import cufflinks as cf
cf.go_offline()
import quantstats as qs
qs.extend_pandas()

In [3]:
font_path_microsoft = r'C:\Users\User\OneDrive\文件\medina\simhei.ttf'
font_path_arial = r'C:\Users\User\OneDrive\文件\medina\Arial Unicode MS.ttf'

if os.path.isfile(font_path_microsoft):
    matplotlib.font_manager.fontManager.addfont(font_path_microsoft)
if os.path.isfile(font_path_arial):
    matplotlib.font_manager.fontManager.addfont(font_path_arial)
matplotlib.rc('font', family='sans-serif')

os.environ['TEJAPI_KEY'] = 'SZf1BjNEcKQhvQmn96eLrNL60Q2RH1'
os.environ['TEJAPI_BASE'] = 'https://api.tej.com.tw'
tejapi.ApiConfig.api_base = 'https://api.tej.com.tw'
tejapi.ApiConfig.api_key = 'SZf1BjNEcKQhvQmn96eLrNL60Q2RH1'
tejapi.ApiConfig.ignoretz = True

In [4]:
import sys
sys.path.append("C:/Users/User/我的雲端硬碟 (owen.lin@mutual-boost.com)/MBQ_Tej_v2/MBQ_tej_v2")
import MBQ_tej_v2_manager
import config
import Tool
import operators_v4

In [5]:
class Handler(dict):
    def __init__(self, path: str = 'Handler_cache2'):
        self.path = path
        os.makedirs(self.path, exist_ok=True)
    
    def __getitem__(self, key):
        file_path = os.path.join(self.path, f'{key}.pkl')
        if os.path.exists(file_path):
            try:
                with open(file_path, 'rb') as f:
                    return pickle.load(f)
            except (pickle.PickleError, EOFError) as e:
                raise KeyError(f"Failed to load key '{key}': {e}")
        else:
            raise KeyError(f"Key '{key}' not found.")
    def __setitem__(self, key, value):
        file_path = os.path.join(self.path, f'{key}.pkl')
        with open(file_path, 'wb') as f:
            pickle.dump(value, f)

    def cash_list(self):
        pkl_set = set(filter(lambda X:X.endswith(".pkl"),os.listdir(self.path)))
        return list(map(lambda X:X[:-4],list(pkl_set)))

    @property
    def info(self):
        return {
            'cache_path': self.path,
            'cache_data_numbers': len(self.cash_list()),
        }

    def _repr_html_(self):
        html = "<table>"
        html += "<tr><th>Key</th><th>Value</th></tr>"
        for key, value in self.info.items():
            html += f"<tr><td>{key}</td><td>{value}</td></tr>"
        html += "</table>"
        return html

In [6]:
MBQ_tej_v2_Handler = MBQ_tej_v2_manager.Handler()
Handler = Handler()

In [7]:
# exec_env = Handler.copy()
# exec_env.update(operators_v4.Alpha_F)

# for factor_name, expr in tqdm.tqdm(config.expr_dict.items()):
#     if factor_name in config.base_factor_expr + config.extra_expr:
#         if factor_name in ['atten_fg', 'disp_fg', 'sbadt_fg']:
#             value = MBQ_tej_v2_Handler[factor_name] == 'Y' 
    
#     if factor_name in config.base_factor_expr:
#         value = MBQ_tej_v2_Handler[factor_name]
#         Handler[factor_name] = value
#         exec_env[factor_name] = value
#     else:

#         try:
#             Handler[factor_name] = eval(expr, exec_env)
#             exec_env[factor_name] = Handler[factor_name]
#         except Exception as e:
#             print(f"無法計算因子 {factor_name}：{e}")

# # 將結果轉為 dict
# factor_dict = {factor_name: Handler[factor_name] for factor_name in config.expr_dict if factor_name not in config.extra_expr}

In [8]:
Adjust_Factor = Handler['Adjust_Factor']
#定义调整股价
Handler['Adj_Open'] = 1 * Handler['Open']
Handler['Adj_Close'] = 1 * Handler['Close']

In [9]:
# con_1 = (Data_dict['turnover']).rolling(5).mean()>=0.2
con_1 = (MBQ_tej_v2_Handler('amt')).rolling(20).mean()>= 50000000
con_2 = MBQ_tej_v2_Handler('sbadt_fg') != 'Y'
# con_3 = MBQ_tej_v2_Handler('disp_fg') != 'Y'
con = con_1 & con_2 

TWN/APIPRCD
amt
C:\Users\User/Documents\MBQ_tej_v2_DB\TWN/APIPRCD\amt.parquet
Custom
Common_Stock
C:\Users\User/Documents\MBQ_tej_v2_DB\Custom\Common_Stock.parquet
分布式讀取:C:\Users\User/Documents\MBQ_tej_v2_DB\Custom\Common_Stock.parquet
Start to update雲端 amt from 2025-04-18 00:00:00 to 2025-04-18 00:00:00
tej_fastget:TWN/APIPRCD:amt
分布式更新:C:\Users\User\我的雲端硬碟 (owen.lin@mutual-boost.com)\MBQ_Tej_v2\MBQ_tej_v2\TWN/APIPRCD\amt


Processing files: 100%|██████████| 1/1 [00:00<00:00,  2.78it/s]


Start to update本地 amt from 2025-04-18 00:00:00 to 2025-04-18 00:00:00
分布式讀取:C:\Users\User\我的雲端硬碟 (owen.lin@mutual-boost.com)\MBQ_Tej_v2\MBQ_tej_v2\TWN/APIPRCD\amt
分布式讀取:C:\Users\User/Documents\MBQ_tej_v2_DB\TWN/APIPRCD\amt.parquet
TWN/APISTKATTR
sbadt_fg
C:\Users\User/Documents\MBQ_tej_v2_DB\TWN/APISTKATTR\sbadt_fg.parquet
Custom
Common_Stock
C:\Users\User/Documents\MBQ_tej_v2_DB\Custom\Common_Stock.parquet
分布式讀取:C:\Users\User/Documents\MBQ_tej_v2_DB\Custom\Common_Stock.parquet
Start to update雲端 sbadt_fg from 2025-04-18 00:00:00 to 2025-04-18 00:00:00
tej_fastget:TWN/APISTKATTR:sbadt_fg
分布式更新:C:\Users\User\我的雲端硬碟 (owen.lin@mutual-boost.com)\MBQ_Tej_v2\MBQ_tej_v2\TWN/APISTKATTR\sbadt_fg


Processing files: 100%|██████████| 1/1 [00:00<00:00,  8.44it/s]

Start to update本地 sbadt_fg from 2025-04-18 00:00:00 to 2025-04-18 00:00:00
分布式讀取:C:\Users\User\我的雲端硬碟 (owen.lin@mutual-boost.com)\MBQ_Tej_v2\MBQ_tej_v2\TWN/APISTKATTR\sbadt_fg


分布式讀取:C:\Users\User/Documents\MBQ_tej_v2_DB\TWN/APISTKATTR\sbadt_fg.parquet


In [10]:
factor_df = pd.concat({data_name:Handler[data_name][con].loc['2015':].stack(future_stack=True) for data_name in tqdm.tqdm(list(Handler.cash_list()))},axis=1)
factor_df5 = pd.read_pickle('./Data/factor_df3.pkl')
factor_df = factor_df.reindex(columns=factor_df5.columns)
# factor_df = factor_df5
factor_df.to_pickle('./Data/factor_df3.pkl')
factor_df

100%|██████████| 196/196 [04:42<00:00,  1.44s/it]


PLUS_deTurn       SPS  Alpha038  Alpha084  Turn20_dePLUS  \
mdate      coid                                                             
2015-01-05 1101    -0.005072  1.200662 -0.053963  1.000000       0.184709   
           1102    -0.013353  1.358317 -0.011163  1.054248       0.067107   
           1103          NaN       NaN       NaN       NaN            NaN   
           1104          NaN       NaN       NaN       NaN            NaN   
           1107          NaN       NaN       NaN       NaN            NaN   
...                      ...       ...       ...       ...            ...   
2025-04-18 9951          NaN       NaN       NaN       NaN            NaN   
           9955          NaN       NaN       NaN       NaN            NaN   
           9958    -0.036376  2.057451 -0.016756  1.000000       1.219371   
           9960          NaN       NaN       NaN       NaN            NaN   
           9962          NaN       NaN       NaN       NaN            NaN   

                 RPV_deTurn20  Alpha001  overnight_EMA5  Alpha039  Alpha046  \
mdate      coid                                                               
2015-01-05 1101     -2.927934 -0.111797        0.000144 -0.609874     -1.00   
           1102     -2.491627  0.032579       -0.000972 -0.466837      0.75   
           1103           NaN       NaN             NaN       NaN       NaN   
           1104           NaN       NaN             NaN       NaN       NaN   
           1107           NaN       NaN             NaN       NaN       NaN   
...                       ...       ...             ...       ...       ...   
2025-04-18 9951           NaN       NaN             NaN       NaN       NaN   
           9955           NaN       NaN             NaN       NaN       NaN   
           9958      1.902790  0.252688        0.007015 -1.529257      5.50   
           9960           NaN       NaN             NaN       NaN       NaN   
           9962           NaN       NaN             NaN       NaN       NaN   

                 ...        Vwap  daily_reverse_EMASTD20  Alpha029  \
mdate      coid  ...                                                 
2015-01-05 1101  ...   43.013553                0.009922  1.193174   
           1102  ...   38.347156                0.007908  0.972014   
           1103  ...         NaN                     NaN       NaN   
           1104  ...         NaN                     NaN       NaN   
           1107  ...         NaN                     NaN       NaN   
...              ...         ...                     ...       ...   
2025-04-18 9951  ...         NaN                     NaN       NaN   
           9955  ...         NaN                     NaN       NaN   
           9958  ...  179.071065                0.051907  0.219220   
           9960  ...         NaN                     NaN       NaN   
           9962  ...         NaN                     NaN       NaN   

                 overnight_EMA_Corr252       CCV        mktcap  Alpha057  \
mdate      coid                                                            
2015-01-05 1101               0.477129 -0.413139  1.587636e+11  0.543507   
           1102               0.435880 -0.495747  1.285754e+11  2.166679   
           1103                    NaN       NaN           NaN       NaN   
           1104                    NaN       NaN           NaN       NaN   
           1107                    NaN       NaN           NaN       NaN   
...                                ...       ...           ...       ...   
2025-04-18 9951                    NaN       NaN           NaN       NaN   
           9955                    NaN       NaN           NaN       NaN   
           9958               0.905637 -0.222346  4.534226e+10  6.142306   
           9960                    NaN       NaN           NaN       NaN   
           9962                    NaN       NaN           NaN       NaN   

                 overnight_EMASTD5  daily_reverse_EMA252  Turn20_mul_PLUS  
mdate      

In [11]:
lend = pd.read_csv("Q2.csv")
lend['coid'] = lend['Ticker'].str.split().str[0]
可借券名單 = lend['coid'].astype(str).tolist()
exp_ret = (Handler['Adj_Close'].shift(-1) / Handler['Adj_Open'].shift(-1) - 1).loc['2015':]

### Catboost

In [12]:
重新訓練模型 = False
if 重新訓練模型:
    print('准备cat需求数据')
    time_slice = '2020-01-01'
    start_date = exp_ret.index[0].date()
    end_date = time_slice
    target = (exp_ret.rank(axis=1, pct=True) * 100).round().stack().loc[f'{start_date}':f'{end_date}'].astype(int)
    X = factor_df.loc[target.index].dropna(how='all')
    y = target.loc[X.index]
    group_id = y.index.get_level_values(0).to_series().index.astype('category').codes
    print('开始训练cat')
    # 初始化 CatBoostRanker 模型
    model = CatBoostRanker(
        task_type="GPU",
        verbose=True,
        iterations = 5000,
        depth = 8,
        learning_rate=0.06,  # 學習率
        l2_leaf_reg = 10,
        random_strength = 3,
        
    )
    model.fit(Pool(data = X,label = y,group_id = group_id),early_stopping_rounds=50)
    print('cat存档')
    model.save_model('cat_model')
    feature_importance = model.get_feature_importance(Pool(X, label=y, group_id=group_id))
    feature_names = X.columns

In [13]:
# from catboost import CatBoostRanker, Pool
# import pandas as pd

# def train_cat_model(
#     exp_ret: pd.DataFrame,
#     factor_df: pd.DataFrame,
#     con: pd.DataFrame,
#     insample_start_date: str,
#     insample_end_date: str,
#     val_start_date: str,
#     val_end_date: str,
#     model_save_path: str,
#     verbose: bool = True,
#     task_type: str = 'GPU',
#     **args
# ):
#     print('準備 CatBoost 排名訓練資料')

#     # -------- 訓練資料 --------
#     target = (
#         exp_ret[con]
#         .rank(axis=1, pct=True)
#         .mul(100)
#         .round()
#         .stack()
#         .loc[insample_start_date:insample_end_date]
#         .astype(int)
#     )

#     X = factor_df.loc[target.index].dropna(how='all')
#     y = target.loc[X.index]
#     group_id = y.index.get_level_values(0).astype('category').codes

#     # -------- 驗證資料 --------
#     val_target = (
#         exp_ret[con]
#         .rank(axis=1, pct=True)
#         .mul(100)
#         .round()
#         .stack()
#         .loc[val_start_date:val_end_date]
#         .astype(int)
#     )

#     val_X = factor_df.loc[val_target.index].dropna(how='all')
#     val_y = val_target.loc[val_X.index]
#     val_group_id = val_y.index.get_level_values(0).astype('category').codes

#     print('開始訓練 CatBoostRanker')
#     model = CatBoostRanker(
#         task_type=task_type,
#         verbose=verbose,
#         use_best_model=True,
#         **args,
#     )

#     model.fit(
#         Pool(data=X, label=y, group_id=group_id),
#         eval_set=Pool(data=val_X, label=val_y, group_id=val_group_id),
#         early_stopping_rounds=50
#     )

#     print(f'模型已儲存至: {model_save_path}')
#     model.save_model(model_save_path)

#     return model


In [14]:
# 訓練模型 = True
# if 訓練模型:
#     os.makedirs('cat_model.v2', exist_ok=True)

#     for year in tqdm.tqdm(range(2019, 2025)):
#         print(f"訓練年份：{year}，模型將用於 {year+1}")

#         insample_start = f'{year-5}-01-01'
#         insample_end   = f'{year-1}-12-31'
#         val_start      = f'{year}-01-01'
#         val_end        = f'{year}-12-31'
#         model_path     = f'./cat_model.v2/{year+1}.cat'

#         train_cat_model(
#             exp_ret=exp_ret,
#             factor_df=factor_df,
#             con=con,  # 相當於原本的 mega_factor.con
#             insample_start_date=insample_start,
#             insample_end_date=insample_end,
#             val_start_date=val_start,
#             val_end_date=val_end,
#             model_save_path=model_path,
#             verbose=False,
#             task_type='GPU',
#         )

In [15]:
# factor_list = list()
# for year in tqdm.tqdm(range(2019,2025)):
#     mega_factor = Mega_Factor_Model.Model(Handler,Handler.cash_list())
#     mega_factor._factor_df = factor_df
#     mega_factor.cat_model_save_path = f'./cat_model.v2/{year+1}.cat'
#     factor_list.append(mega_factor.predict().loc[f'{year+1}':f'{year+1}'])
#     #factor_list.append(mega_factor.predict().loc[f'{year+1}':])
# factor2 = pd.concat(factor_list,axis=1).groupby('coid',axis='columns').mean()

In [16]:
from catboost import CatBoostRanker, Pool
# 從文件加載模型
loaded_ranker = CatBoostRanker()
cat_model_path = 'cat_model'
loaded_ranker.load_model(cat_model_path)
print("模型已加載。")
# 使用加載的模型進行預測
y_pred_loaded = loaded_ranker.predict(factor_df)
print("加載模型的預測完成。")
factor = pd.Series(y_pred_loaded,index =  factor_df.index).unstack()

模型已加載。
加載模型的預測完成。


# 因子分析

### 這裡不用跑

In [17]:
def calc_IC_series(factor: pd.DataFrame, exp_ret: pd.DataFrame) -> pd.Series:
    IC_series = factor.corrwith(exp_ret, axis=1, method='spearman')
    return IC_series.sort_index()

def run_IC_test(IC_series: pd.Series, insample_start_date: str, insample_end_date: str) -> pd.DataFrame:
    insample_start = pd.to_datetime(insample_start_date)
    insample_end = pd.to_datetime(insample_end_date)
    outsample_start = pd.to_datetime("2020-01-01")

    IC_mean = pd.Series({
        f"樣本內({insample_start.date()}~{insample_end.date()})": IC_series.loc[insample_start:insample_end].mean(),
        f"樣本外({outsample_start.date()}~{IC_series.index[-1].date()})": IC_series.loc[outsample_start:].mean(),
    })

    ICIR = pd.Series({
        f"樣本內({insample_start.date()}~{insample_end.date()})": IC_mean[0] / IC_series.loc[insample_start:insample_end].std(),
        f"樣本外({outsample_start.date()}~{IC_series.index[-1].date()})": IC_mean[1] / IC_series.loc[outsample_start:].std(),
    })

    return pd.concat({"IC": IC_mean, "ICIR": ICIR}, axis=1)

In [18]:
因子檢定模式 = True
if 因子檢定模式:
    IC_Se = calc_IC_series(factor[con], exp_ret)
    display(run_IC_test(IC_Se, insample_start_date='2015-01-01', insample_end_date='2019-12-31'))
    Tool.backtest_factor(factor[con], exp_ret, rank_range_n=10, start_date='2019-12-31')
    IC_Se.ewm(span=60).mean().dropna().iloc[60:].iplot(hline=[IC_Se.mean()])

,IC,ICIR
樣本內(2015-01-01~2019-12-31),0.366525,3.030141
樣本外(2020-01-01~2025-04-18),0.125676,0.965192


IC_mean:0.1258
IC_IR:0.966


100%|██████████| 10/10 [00:01<00:00,  6.87it/s]


,CAGR(%),Sharpe,Calmar,MDD(%),單利MDD(%),样本胜率(%),周胜率(%),月胜率(%),年胜率(%),盈亏比(avg_win/avg_loss),总赚赔比(profit_factor),预期报酬(bps),样本数
0% ~ 10%,-53.86,-9.79,-0.54,-99.74,-594.40,27.20,8.66,0.00,0.00,0.51,0.19,-46.16,1283
10% ~ 20%,-42.51,-8.13,-0.43,-98.57,-422.07,30.79,11.19,0.00,0.00,0.58,0.26,-33.06,1283
20% ~ 30%,-31.51,-6.04,-0.33,-94.60,-291.52,36.87,17.69,4.62,0.00,0.64,0.37,-22.62,1283
30% ~ 40%,-25.83,-5.01,-0.29,-89.99,-226.53,38.58,22.74,4.62,14.29,0.71,0.45,-17.86,1283
40% ~ 50%,-19.27,-3.99,-0.24,-81.24,-164.95,40.30,27.80,15.38,14.29,0.77,0.52,-12.79,1283
50% ~ 60%,-13.94,-2.81,-0.20,-69.97,-115.67,42.40,33.21,15.38,14.29,0.86,0.63,-8.98,1283
60% ~ 70%,-9.00,-1.87,-0.17,-54.38,-74.21,44.27,35.74,32.31,14.29,0.92,0.73,-5.64,1283
70% ~ 80%,3.39,0.74,0.45,-7.50,-7.15,50.35,50.90,63.08,71.43,1.12,1.14,1.99,1283
80% ~ 90%,15.91,2.90,1.97,-8.07,-5.70,57.83,68.23,84.62,100.00,1.25,1.71,8.83,1283
90% ~ 100%,38.13,4.44,4.28,-8.91,-7.76,64.07,77.62,89.23,100.00,1.30,2.31,19.34,1283


In [19]:
def max_drawdown(prices):
    cumulative_max = prices.cummax()
    drawdown = (prices - cumulative_max) / cumulative_max
    mdd = drawdown.min()
    return mdd

In [20]:
from datetime import datetime
benchmark = tejapi.get('TWN/APIPRCD', coid='IX0001',opts={'columns': ['mdate','roi']}, mdate={'gte': '2010-01-01', 'lte':Adjust_Factor.index[-1]}, paginate=True)
benchmark["mdate"] = pd.to_datetime(benchmark["mdate"])
benchmark["mdate"] = benchmark["mdate"].dt.strftime("%Y-%m-%d")
benchmark_return = benchmark['roi'] / 100
benchmark_return.index = Adjust_Factor.index
bench_exp_ret = benchmark_return.shift(-1)

# 估計權重
score 用來估計可借券名單的權重，這裡還是使用成交金額

In [21]:
span_list = [5, 20, 60, 120, 252, 500, 1000, 2000, 3000, 4000]
weights = [1, 1, 1.2, 1.5, 1.8, 2, 2, 2, 1.5, 1.2]
f = 3
最短上市時長 = 60
start_date = '2020-01-01'
上市時長 = Handler['Close'].notna().astype(int).expanding().sum()
def get_short_score(span, f=1):
    mean = exp_ret.shift().ewm(span=span, min_periods=1).mean()
    std = exp_ret.shift().ewm(span=span, min_periods=1).std()
    sharpe = mean / std**f
    valid = 上市時長 > 最短上市時長
    return sharpe.where((sharpe < 0) & valid).fillna(0)

score = sum(w * get_short_score(span, f) for span, w in zip(span_list, weights))

### 回測

In [22]:
def get_tick(price):
    if price <= 10:
        return 0.01
    elif price <= 49.95:
        return 0.05
    elif price <= 99.9:
        return 0.1
    elif price <= 499.5:
        return 0.5
    elif price <= 999:
        return 1
    else:
        return 5

def calculate_return_series(
    ret,
    Low_N,
    AUM,
    adj_close,
    exp_ret,
    bench_exp_ret,
):
    ret = ret.loc['2020':].fillna(0)
    ranking_asc = ret.rank(axis=1, ascending=True, method='first')
    short_signal = (ranking_asc <= Low_N).astype(int) * -1

    成交金額 = Handler['Value_Dollars'].rolling(20).mean().loc['2020':]
    W = short_signal * 成交金額
    W_sum = abs(W.sum(axis=1))
    W = W.div(W_sum.where(W_sum != 0, np.nan), axis=0)

    adj_close = adj_close.replace({0: np.nan}) 
    weighting = np.floor(((W * AUM) / (1000 * adj_close))) * (1000 * adj_close) / AUM
    weighting.replace({0: np.nan}, inplace=True)

    # 計算每日組合損益
    def calculate_return(wgt):
        delta_weighting = wgt.copy()
        day_buy_fee  = delta_weighting[delta_weighting > 0].abs() * (0.001425 * 0.3 + 0.003 / 2)
        day_sell_fee = delta_weighting[delta_weighting < 0].abs() * (0.001425 * 0.3)

        tick_cost = adj_close.applymap(get_tick) * 1  # tick 單位 × 來回
        shares_traded  = np.floor((delta_weighting.abs() * AUM) / (1000 * adj_close)) * 1000  # 換算出張數（實際股數）
        slippage_cash = shares_traded * tick_cost  # 每筆滑價成本 = 股數 × tick 單位
        slippage_total = slippage_cash.sum(axis=1)
        # 算滑價對報酬的影響 除以實際資金使用
        capital_used = (abs(wgt) * AUM).sum(axis=1).replace(0, np.nan)
        slippage_return = (slippage_total / capital_used).fillna(0)
        return (wgt * exp_ret).sum(axis=1) - day_buy_fee.fillna(0).sum(axis=1) - day_sell_fee.fillna(0).sum(axis=1) - slippage_return

    short_return = calculate_return(weighting)

    bt_ret = pd.concat({
        'short_return': short_return,
        'benchmark': bench_exp_ret.loc[exp_ret.index.min():exp_ret.index.max()].reindex(exp_ret.index).fillna(method='ffill').fillna(0)
    }, axis=1).loc["2020-01-02":]

    return bt_ret, weighting


In [23]:
bt_ret, weightings = calculate_return_series(
    ret=factor[con],
    Low_N=20,
    AUM = 50000000,
    adj_close=Handler['Adj_Close'],
    exp_ret=exp_ret,
    bench_exp_ret=bench_exp_ret)

In [24]:
import warnings
warnings.filterwarnings("ignore")
display(pd.concat({"CAGR(%)":bt_ret.cagr() * 100,
        'Sharpe' : bt_ret.mean() / bt_ret.std()*252**0.5,
        'Sortino': bt_ret.mean() / bt_ret[bt_ret < 0].std()*252**0.5,
        'Calmar':bt_ret.cagr() / abs(bt_ret.max_drawdown()),
        'MDD(%)' : bt_ret.max_drawdown()*100,
        '單利MDD(%)' : max_drawdown(bt_ret.cumsum().add(1))*100,
        '樣本勝率(%)' : bt_ret.apply(lambda X:((X.dropna()>0).sum() / X.dropna().shape[0])*100),
        '周勝率(%)' : bt_ret.apply(lambda X:((X.dropna().add(1).resample('W').prod().sub(1)>0).sum() / X.dropna().add(1).resample('W').prod().sub(1).dropna().shape[0])*100),
        '月勝率(%)' : bt_ret.apply(lambda X:((X.dropna().add(1).resample('ME').prod().sub(1)>0).sum() / X.dropna().add(1).resample('ME').prod().sub(1).shape[0])*100),
        '年勝率(%)' : bt_ret.apply(lambda X:((X.dropna().add(1).resample('YE').prod().sub(1)>0).sum() / X.dropna().add(1).resample('YE').prod().sub(1).shape[0])*100),
        '預期報酬(bps)':((1 + bt_ret).prod() ** (1 / len(bt_ret)) - 1)*10000,
        '平均交易報酬(%)': bt_ret.apply(lambda x: (x[x != 0].mean()) * 100),
        'avg_win/avg_loss' : bt_ret.apply(lambda X:(X[X > 0].mean() / abs(X[X < 0].mean()))),
        'profit_factor' : bt_ret.profit_factor(),
        },axis = 1).round(2))
bt_ret.cumsum().ffill().iplot()
bt_ret.to_drawdown_series().iplot(title = 'Drawdown')
pd.concat({
    f'({round(weightings.loc["2020-01-02":].sum(axis=1).mean(), 4)})': -1 * weightings.loc["2020-01-02":].sum(axis=1),
}, axis=1).iplot(title='現金使用率')

,CAGR(%),Sharpe,Sortino,Calmar,MDD(%),單利MDD(%),樣本勝率(%),周勝率(%),月勝率(%),年勝率(%),預期報酬(bps),平均交易報酬(%),avg_win/avg_loss,profit_factor
short_return,123.85,4.32,7.82,6.07,-20.39,-5.16,59.70,71.12,84.38,100.00,48.29,0.50,1.41,2.08
benchmark,6.38,0.57,0.70,0.20,-31.63,-32.82,54.79,57.04,57.81,66.67,3.70,0.04,0.92,1.11


In [25]:
bt_ret_last10days = bt_ret.iloc[-55:-1] * 100
bt_ret_last10days.iloc[0] = 0
bt_ret_last10days.cumsum().iplot(title='累積報酬 % ')

In [26]:
ind= MBQ_tej_v2_Handler('Industry')[con]
mkt = MBQ_tej_v2_Handler('mkt_bd_e')[con]
ind = ind.stack().reset_index().rename(columns={0: '產業別'})
mkt = mkt.stack().reset_index().rename(columns={0: '市場別'})

TWN/APISTKATTR
main_ind_c
C:\Users\User/Documents\MBQ_tej_v2_DB\TWN/APISTKATTR\main_ind_c.parquet
Custom
Common_Stock
C:\Users\User/Documents\MBQ_tej_v2_DB\Custom\Common_Stock.parquet
分布式讀取:C:\Users\User/Documents\MBQ_tej_v2_DB\Custom\Common_Stock.parquet
Start to update雲端 main_ind_c from 2025-04-17 00:00:00 to 2025-04-18 00:00:00
tej_fastget:TWN/APISTKATTR:main_ind_c
分布式更新:C:\Users\User\我的雲端硬碟 (owen.lin@mutual-boost.com)\MBQ_Tej_v2\MBQ_tej_v2\TWN/APISTKATTR\main_ind_c


Processing files: 100%|██████████| 1/1 [00:00<00:00,  9.05it/s]

Start to update本地 main_ind_c from 2025-04-17 00:00:00 to 2025-04-18 00:00:00
分布式讀取:C:\Users\User\我的雲端硬碟 (owen.lin@mutual-boost.com)\MBQ_Tej_v2\MBQ_tej_v2\TWN/APISTKATTR\main_ind_c


分布式讀取:C:\Users\User/Documents\MBQ_tej_v2_DB\TWN/APISTKATTR\main_ind_c.parquet
TWN/APISTKATTR
mkt_bd_e
C:\Users\User/Documents\MBQ_tej_v2_DB\TWN/APISTKATTR\mkt_bd_e.parquet
Custom
Common_Stock
C:\Users\User/Documents\MBQ_tej_v2_DB\Custom\Common_Stock.parquet
分布式讀取:C:\Users\User/Documents\MBQ_tej_v2_DB\Custom\Common_Stock.parquet
Start to update雲端 mkt_bd_e from 2025-04-17 00:00:00 to 2025-04-18 00:00:00
tej_fastget:TWN/APISTKATTR:mkt_bd_e
分布式更新:C:\Users\User\我的雲端硬碟 (owen.lin@mutual-boost.com)\MBQ_Tej_v2\MBQ_tej_v2\TWN/APISTKATTR\mkt_bd_e


Processing files: 100%|██████████| 1/1 [00:00<00:00, 17.86it/s]

Start to update本地 mkt_bd_e from 2025-04-17 00:00:00 to 2025-04-18 00:00:00
分布式讀取:C:\Users\User\我的雲端硬碟 (owen.lin@mutual-boost.com)\MBQ_Tej_v2\MBQ_tej_v2\TWN/APISTKATTR\mkt_bd_e


分布式讀取:C:\Users\User/Documents\MBQ_tej_v2_DB\TWN/APISTKATTR\mkt_bd_e.parquet


### 訊號

In [27]:
import pandas as pd

# === 基本設定 ===
date = pd.to_datetime("2025-4-10")
low_n = 20
刪除標的 = []
指定交易标的 = []
有限購入張數 = {}

last_factor = factor[con].loc[date].drop(columns=刪除標的).dropna().sort_values(ascending=True)
last_factor = last_factor[last_factor.index.isin(可借券名單)]
singal_bool2 = last_factor.rank(ascending=True, method='first') <= low_n
singal_list2 = singal_bool2[singal_bool2].index.to_list()

Close = Handler['Close']
volume = Handler['Value_Dollars']
volume = (volume / 1e8).round(2)
收盤價2 = Close.drop(columns=刪除標的).loc[date][singal_list2]
成交量2 = volume.drop(columns=刪除標的)[singal_list2].rolling(20).mean().loc[date].round(2)

date_idx = Close.index.get_loc(date)
one_week_ago = Close.index[date_idx - 5]
一週前價 = Close.loc[one_week_ago, singal_list2]
今日價 = 收盤價2
近一週報酬率 = (今日價 / 一週前價 - 1).round(3) * 100
ind_ = ind[ind['mdate'] == date].set_index('coid')['產業別'].str.replace(r'^[A-Za-z0-9\s]+', '', regex=True)
產業別2 = ind_.reindex(singal_list2)
市場別2 = mkt[mkt['mdate'] == date].set_index('coid')['市場別'].reindex(singal_list2)
上市天數 = Close[singal_list2].notna().astype(int).sum()
上市天數.name = '上市上櫃天數'


是否可借券 = pd.Series(
    [coid in 可借券名單 for coid in singal_list2],
    index=singal_list2,
    name='是否可借券'
).map({True: 'Y', False: 'F'})
lend_qty = (lend.set_index('coid')['Qty'] // 1000).loc[singal_list2]  # 換算成張數（整數除法）


# === 顯示結果 ===
成交金額_加總 = 成交量2.loc[singal_list2].abs().sum()
訊號表 = pd.concat([
    收盤價2.rename('收盤價'),
    (收盤價2 / 1.095).rename('跌停價'),
    成交量2.rename('成交金額(億元)'),
    近一週報酬率.rename('近一週報酬率%'),
    產業別2.rename('產業別'),
    市場別2.rename('市場別'),
    上市天數,
    是否可借券.rename('是否可借券'),
    lend_qty.rename('可借券張數'),
], axis=1)

print(f'訊號日期: {date.date()}')
# print(f"💡 可轉債報酬率（{date.date()}）：{(cb_return * 100):.2f}%")
display(訊號表)

訊號日期: 2025-04-10


,收盤價,跌停價,成交金額(億元),近一週報酬率%,產業別,市場別,上市上櫃天數,是否可借券,可借券張數
8422,186.00,169.863014,0.96,-3.4,綠能環保,TSE,3312,Y,24
2845,12.85,11.735160,1.04,-4.8,金融業,TSE,3748,Y,766
8072,31.25,28.538813,0.92,7.8,電子工業,TSE,3748,Y,6
2809,48.85,44.611872,1.10,-1.1,金融業,TSE,3748,Y,105
2886,38.05,34.748858,8.74,-4.2,金融業,TSE,3748,Y,1600
2834,14.15,12.922374,3.14,-4.1,金融業,TSE,3748,Y,5094
2633,26.50,24.200913,1.50,-2.0,航運業,TSE,2065,Y,410
2851,26.10,23.835616,0.67,-4.9,金融業,TSE,3748,Y,79
5251,31.80,29.041096,0.64,11.0,電子類,OTC,3026,Y,7
2887,15.60,14.246575,7.48,-10.1,金融業,TSE,3748,Y,558


# 確定訊號

In [28]:
target_date = pd.Timestamp("2025-04-11")
Low_N = 20

actual_weight = weightings.loc[target_date]
actual_set = set(actual_weight[actual_weight < 0].dropna().index)

factor1 = factor[con].loc[target_date]
ranking_asc = factor1.rank(ascending=True, method='first')
short_signal = (ranking_asc <= Low_N).astype(int) * -1

final_signal = short_signal[(short_signal != 0)]
expected_weight = weightings.loc[target_date].reindex(final_signal.index)

expected_set = set(expected_weight[expected_weight < 0].dropna().index)
diff = actual_set.symmetric_difference(expected_set)
print(f"持股數量: {len(actual_set)}，應持有: {len(expected_set)}，不一致數量: {len(diff)}")
if diff:
    print("以下為不一致標的：")
    for stock in sorted(diff):
        print(stock)
else:
    print("完全一致")


持股數量: 20，應持有: 20，不一致數量: 0
完全一致


In [29]:
績效回顧 = bt_ret.reindex(exp_ret.index)
display((績效回顧.iloc[-21:]*100).round(2))

,short_return,benchmark
mdate,,
2025-03-19,-0.04,1.90
2025-03-20,1.95,-0.75
2025-03-21,2.84,-0.46
2025-03-24,2.86,0.75
2025-03-25,-0.23,-0.06
2025-03-26,1.06,-1.39
2025-03-27,2.51,-1.59
2025-03-28,2.94,-4.20
2025-03-31,1.11,2.82


# Max Drawdown

In [30]:
drawdown_periods = [
    {'name': 'Period 1', 'start': '2022-06-30', 'end': '2022-09-05'},
    # {'name': 'Period 2', 'start': '2022-10-28', 'end': '2022-12-06	'},
]

def calculate_metrics(data):
    display(pd.concat({"CAGR(%)":data.cagr() * 100,
        'Sharpe' : data.mean() / data.std()*252**0.5,
        'Sortino': data.mean() / data[data < 0].std()*252**0.5,
        'Calmar':data.cagr() / abs(data.max_drawdown()),
        'MDD(%)' : data.max_drawdown()*100,
        '樣本勝率(%)' : data.apply(lambda X:((X.dropna()>0).sum() / X.dropna().shape[0])*100),
        '周勝率(%)' : data.apply(lambda X:((X.dropna().add(1).resample('W').prod().sub(1)>0).sum() / X.dropna().add(1).resample('W').prod().sub(1).dropna().shape[0])*100),
        '月勝率(%)' : data.apply(lambda X:((X.dropna().add(1).resample('ME').prod().sub(1)>0).sum() / X.dropna().add(1).resample('ME').prod().sub(1).shape[0])*100),
        '年勝率(%)' : data.apply(lambda X:((X.dropna().add(1).resample('YE').prod().sub(1)>0).sum() / X.dropna().add(1).resample('YE').prod().sub(1).shape[0])*100),
        '預期報酬(bps)':((1 + data).prod() ** (1 / len(data)) - 1)*10000,
        "avg_win(%)": data.apply(lambda X: X[X > 0].mean() * 100),
        "avg_loss(%)": data.apply(lambda X: X[X < 0].mean() * 100),
        'avg_win/avg_loss' : data.apply(lambda X:(X[X > 0].mean() / abs(X[X < 0].mean()))),
        '平均交易報酬(%)': data.apply(lambda x: (x[x != 0].mean()) * 100),
        'profit_factor' : data.profit_factor(),
        },axis = 1).round(2)) 

for period in drawdown_periods:
    period_data = bt_ret.loc[period['start']:period['end']]
    # period_data = period_data.drop(columns=['ls_return'])
    print(f"Performance Metrics for {period['start']} to {period['end']}")
    metrics = calculate_metrics(period_data)
    display(metrics)

cumulative_returns = bt_ret[['short_return', 'benchmark']].cumsum().ffill()

for i, period in enumerate(drawdown_periods):
    start, end = period['start'], period['end']

    period_data = cumulative_returns.loc[start:end]
    period_data = period_data - period_data.iloc[0]  
    period_data.iplot()

Performance Metrics for 2022-06-30 to 2022-09-05


,CAGR(%),Sharpe,Sortino,Calmar,MDD(%),樣本勝率(%),周勝率(%),月勝率(%),年勝率(%),預期報酬(bps),avg_win(%),avg_loss(%),avg_win/avg_loss,平均交易報酬(%),profit_factor
short_return,-30.03,-1.38,-2.15,-1.47,-20.39,50.00,45.45,50.0,0.0,-19.76,1.40,-1.76,0.80,-0.18,0.80
benchmark,-3.72,-0.17,-0.22,-0.71,-5.20,54.17,63.64,25.0,0.0,-2.10,0.84,-1.02,0.82,-0.01,0.97


None

### 預測大盤

In [31]:
twse = tejapi.get('TWN/APIPRCD', coid='IX0001',opts={'columns': ['mdate','close_d','open_d','high_d','low_d','vol','turnover']}, mdate={'gte': '2010-01-01', 'lte':'2025-04-18'}, paginate=True)
top50 = tejapi.get('TWN/APIPRCD', coid='0050',opts={'columns': ['mdate','close_d','open_d']}, mdate={'gte': '2010-01-01', 'lte':datetime.today().strftime('%Y-%m-%d')}, paginate=True)

In [32]:
Zscore = lambda series: (series - series.rolling(60).mean()) / series.rolling(60).std()
非負化處理 = lambda series: Zscore(series) + Zscore(series).rolling(60).min().abs()

In [33]:
twse.rename(columns={'mdate':'date','close_d':'Close','open_d':'Open','high_d':'High','low_d':'Low','vol':'Volume','turnover':'Turnover'}, inplace=True)
twse.set_index(['date'], inplace=True)
twse['Adjust_Factor'] = 1
twse['Adj_Open'] = twse['Open'] * twse['Adjust_Factor']
twse['Adj_Close'] = twse['Close'] * twse['Adjust_Factor']
twse['Adj_High'] = twse['High'] * twse['Adjust_Factor']
twse['Adj_Low'] = twse['Low'] * twse['Adjust_Factor']

twse['daily_ret'] = twse['Adj_Close'].pct_change()
top50['daily_ret'] = top50['close_d'].pct_change()
twse['MoM5'] = twse['Adj_Close'].pct_change(5)
twse['MoM20'] = twse['Adj_Close'].pct_change(20)
twse['MoM252'] = twse['Adj_Close'].pct_change(252)

twse['daily_EMA5'] = twse['daily_ret'].ewm(span=5).mean()
twse['daily_EMA20'] = twse['daily_ret'].ewm(span=20).mean()
twse['daily_EMA252'] = twse['daily_ret'].ewm(span=252).mean()
twse['daily_EMASTD5'] = twse['daily_ret'].ewm(span=5).std()
twse['daily_EMASTD20'] = twse['daily_ret'].ewm(span=20).std()
twse['daily_EMASTD252'] = twse['daily_ret'].ewm(span=252).std()
twse['daily_EMA_Sharpe5'] = twse['daily_EMA5'] / twse['daily_EMASTD5']
twse['daily_EMA_Sharpe20'] = twse['daily_EMA20'] / twse['daily_EMASTD20']
twse['daily_EMA_Sharpe252'] = twse['daily_EMA252'] / twse['daily_EMASTD252']
twse['daily_EMA_Sharpe252_plus'] = twse['daily_EMA252']**2 / twse['daily_EMASTD252']

twse['intraday_ret'] = (twse['Adj_Close'] / twse['Adj_Open']) - 1
twse['intraday_EMA5'] = twse['intraday_ret'].ewm(span=5).mean()
twse['intraday_EMA20'] = twse['intraday_ret'].ewm(span=20).mean()
twse['intraday_EMA252'] = twse['intraday_ret'].ewm(span=252).mean()
twse['intraday_EMASTD5'] = twse['intraday_ret'].ewm(span=5).std()
twse['intraday_EMASTD20'] = twse['intraday_ret'].ewm(span=20).std()
twse['intraday_EMASTD252'] = twse['intraday_ret'].ewm(span=252).std()
twse['intraday_EMA_Sharpe5'] = twse['intraday_EMA5'] / twse['intraday_EMASTD5']
twse['intraday_EMA_Sharpe20'] = twse['intraday_EMA20'] / twse['intraday_EMASTD20']
twse['intraday_EMA_Sharpe252'] = twse['intraday_EMA252'] / twse['intraday_EMASTD252']
twse['intraday_EMA_Sharpe252_plus'] = twse['intraday_EMA252']**2 / twse['intraday_EMASTD252']


twse['overnight_ret'] = (twse['Adj_Open'] / twse['Adj_Close'].shift(1)) - 1
twse['overnight_EMA5'] = twse['overnight_ret'].ewm(span=5).mean()
twse['overnight_EMA20'] = twse['overnight_ret'].ewm(span=20).mean()
twse['overnight_EMA252'] = twse['overnight_ret'].ewm(span=252).mean()
twse['overnight_EMASTD5'] = twse['overnight_ret'].ewm(span=5).std()
twse['overnight_EMASTD20'] = twse['overnight_ret'].ewm(span=20).std()
twse['overnight_EMASTD252'] = twse['overnight_ret'].ewm(span=252).std()
twse['overnight_EMA_Sharpe5'] = twse['overnight_EMA5'] / twse['overnight_EMASTD5']
twse['overnight_EMA_Sharpe20'] = twse['overnight_EMA20'] / twse['overnight_EMASTD20']
twse['overnight_EMA_Sharpe252'] = twse['overnight_EMA252'] / twse['overnight_EMASTD252']
twse['overnight_EMA_Sharpe252_plus'] = twse['overnight_EMA252']**2 / twse['overnight_EMASTD252']


twse['daily_reverse_ret'] = twse['overnight_ret'] - twse['intraday_ret']
twse['daily_reverse_EMA5'] = twse['daily_reverse_ret'].ewm(span=5).mean()
twse['daily_reverse_EMA20'] = twse['daily_reverse_ret'].ewm(span=20).mean()
twse['daily_reverse_EMA252'] = twse['daily_reverse_ret'].ewm(span=252).mean()
twse['daily_reverse_EMASTD5'] = twse['daily_reverse_ret'].ewm(span=5).std()
twse['daily_reverse_EMASTD20'] = twse['daily_reverse_ret'].ewm(span=20).std()
twse['daily_reverse_EMASTD252'] = twse['daily_reverse_ret'].ewm(span=252).std()
twse['daily_reverse_EMA_Sharpe5'] = twse['daily_reverse_EMA5'] / twse['daily_reverse_EMASTD5']
twse['daily_reverse_EMA_Sharpe20'] = twse['daily_reverse_EMA20'] / twse['daily_reverse_EMASTD20']
twse['daily_reverse_EMA_Sharpe252'] = twse['daily_reverse_EMA252'] / twse['daily_reverse_EMASTD252']
twse['daily_reverse_EMA_Sharpe252_plus'] = twse['daily_reverse_EMA252']**2 / twse['daily_reverse_EMASTD252']

twse['Turn20'] = twse['Turnover'].rolling(20).mean()
twse['PctTurn'] = twse['Turnover'] / twse['Turnover'].rolling(40).mean().shift(20) - 1
twse['PctTurn20'] = twse['PctTurn'].rolling(20).mean()
twse['振幅'] = (twse['Adj_High'] - twse['Adj_Low']) / twse['Adj_Close'].shift()
twse['振幅換手率因子'] = (twse['PctTurn'] * twse['振幅']).rolling(20).mean()
twse['STR'] = twse['Turnover'].rolling(20).std()
twse['Vol20'] = twse['daily_ret'].rolling(20).std()
twse['每日換手率變化率'] = twse['Turnover'].pct_change()
twse['每日換手率變化率_MA20'] = twse['每日換手率變化率'].rolling(20).mean()
twse['GTR'] = twse['每日換手率變化率'].rolling(20).std()
twse['SCR'] = (twse['STR'] / (twse['Turnover'].rolling(40, min_periods=1).std().shift(20))).shift() - 1
twse['FAC'] = (twse['Adj_High'] - twse['Adj_Low']) / twse['Adj_Close'].shift()
twse['PLUS'] = (2 * twse['Adj_Close'] - twse['Adj_High'] - twse['Adj_Low']) / twse['Adj_Close'].shift()
twse['Turn20_mul_PLUS'] = 非負化處理(Zscore(twse['Turn20'])) * 非負化處理(Zscore(twse['PLUS']))
twse['STR_mul_PLUS'] = 非負化處理(Zscore(twse['STR'])) * 非負化處理(Zscore(twse['PLUS']))
twse['SCR_mul_PLUS'] = 非負化處理(Zscore(twse['SCR'])) * 非負化處理(Zscore(twse['PLUS']))
twse['Turn20_mul_FAC'] = 非負化處理(Zscore(twse['Turn20'])) * 非負化處理(Zscore(twse['FAC']))
twse['STR_mul_FAC'] = 非負化處理(Zscore(twse['STR'])) * 非負化處理(Zscore(twse['FAC']))
twse['SCR_mul_FAC'] = 非負化處理(Zscore(twse['SCR'])) * 非負化處理(Zscore(twse['FAC']))
twse['Turn20_mul_SCR'] = 非負化處理(Zscore(twse['Turn20'])) * 非負化處理(Zscore(twse['SCR']))
twse['STR_mul_SCR'] = 非負化處理(Zscore(twse['STR'])) * 非負化處理(Zscore(twse['SCR']))
twse['CCV'] = twse['Adj_Close'].rolling(20).corr(twse['Turnover'])
twse['CDCV'] = twse['Adj_Close'].diff().rolling(20).corr(twse['Turnover'])
twse['CCOV'] = (twse['Adj_Close'] - twse['Adj_Open']).rolling(20).corr(twse['Turnover'])
twse['COV'] = (twse['Adj_Open'] - twse['Adj_Close'].shift()).rolling(20).corr(twse['Turnover'].shift())
twse['RPV'] = Zscore(twse['CCOV']) - Zscore(twse['COV'])
twse['日内报酬变化量20日指数加权波动度'] = twse['intraday_ret'].pct_change().ewm(span=20).mean()
twse['short_return'] = bt_ret['short_return']
twse['top_50'] = top50['close_d']
twse['ret_spread_5'] = (twse['daily_ret'].rolling(5).mean() - top50['daily_ret'].rolling(5).mean())
twse['ret_spread_20'] = (twse['daily_ret'].rolling(20).mean() - top50['daily_ret'].rolling(20).mean())
twse['ret_spread_252'] = (twse['daily_ret'].rolling(252).mean() - top50['daily_ret'].rolling(252).mean())
twse['corr_20'] = twse['daily_ret'].rolling(20).corr(top50['daily_ret'])
twse['corr_60'] = twse['daily_ret'].rolling(60).corr(top50['daily_ret'])
twse['ema_spread_20'] = twse['Adj_Close'].ewm(span=20).mean() - top50['close_d'].ewm(span=20).mean()
twse['vol_spread_20'] = twse['daily_ret'].rolling(20).std() - top50['daily_ret'].rolling(20).std()
twse['kbar_strength'] = (twse['Adj_Close'] - twse['Adj_Open']) / (twse['Adj_High'] - twse['Adj_Low'] + 1e-6)
twse['gap_percent'] = twse['Adj_Open'] / twse['Adj_Close'].shift(1) - 1
twse['body_to_range'] = abs(twse['Adj_Close'] - twse['Adj_Open']) / (twse['Adj_High'] - twse['Adj_Low'] + 1e-6)

In [34]:
import talib
def calculate_technical_indicators(df):
    df = df.copy()
    close = df['Close']
    high = df['High']
    low = df['Low']
    volume = df['Volume']
    
    # MACD and Signal Line High
    df['MACD'], df['Signal Line'], _ = talib.MACD(close)
    df['MACD_High'] = df[['MACD', 'Signal Line']].max(axis=1)
    
    # MACD Histogram Low
    df['MACD_Histogram'] = talib.MACD(close)[2]
    df['MACD_Histogram_Low'] = df['MACD_Histogram'].min()
    
    # PPO Adj. Close
    df['PPO'] = talib.PPO(close)
    
    # ADX Volume
    df['ADX'] = talib.ADX(high, low, close)
    df['Volume_ADX'] = talib.SMA(volume, timeperiod=14)
    
    # Momentum
    df['Momentum'] = close - close.shift(10)
    
    # CCI
    df['CCI'] = talib.CCI(high, low, close, timeperiod=20)
    
    # ROC
    df['ROC'] = talib.ROC(close, timeperiod=10)
    
    # Stochastic %D and %K
    df['%K'], df['%D'] = talib.STOCH(high, low, close)
    
    # Williams %R
    df['Williams %R'] = talib.WILLR(high, low, close)
    
    # SMA20, SMA50, SMA100
    df['SMA20'] = talib.SMA(close, timeperiod=20)
    df['SMA50'] = talib.SMA(close, timeperiod=50)
    df['SMA100'] = talib.SMA(close, timeperiod=100)
    
    # EMA20, EMA50, EMA100
    df['EMA20'] = talib.EMA(close, timeperiod=20)
    df['EMA50'] = talib.EMA(close, timeperiod=50)
    df['EMA100'] = talib.EMA(close, timeperiod=100)
    
    # Bollinger Bands (Upper, Middle, and Lower Bands)
    df['Middle Band'], _, df['Lower Band'] = talib.BBANDS(close)
    df['Upper Band'] = df['Middle Band'] + 2 * (close.rolling(window=20).std())
    
    # PSAR
    df['PSAR'] = talib.SAR(high, low)
    
    # OBV
    df['OBV'] = talib.OBV(close, volume)
    
    # Chaikin Oscillator
    df['Chaikin Oscillator'] = talib.ADOSC(high, low, close, volume, fastperiod=3, slowperiod=10)
    
    # MFI
    typical_price = (high + low + close) / 3
    raw_money_flow = typical_price * volume
    df['Money Flow Ratio'] = np.where(raw_money_flow.diff() > 0, raw_money_flow, 0)
    df['Money Flow Ratio'] = df['Money Flow Ratio'].rolling(window=14).sum() / raw_money_flow.rolling(window=14).sum()
    df['MFI'] = 100 - 100 / (1 + df['Money Flow Ratio'])
    
    # ATR
    df['TR'] = np.maximum(high - low, high - close.shift(), close.shift() - low)
    df['ATR'] = df['TR'].rolling(window=14).mean()
    
    # RSI
    df['RSI'] = talib.RSI(close)
    
    return df

twse = calculate_technical_indicators(twse)

In [35]:
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import cross_val_score
import hashlib
def snapshot_hash(df):
    return hashlib.md5(pd.util.hash_pandas_object(df, index=True).values).hexdigest()


def create_or_load_snapshot(twse, year, data_dir="snapshot_data", overwrite=False):
    os.makedirs(data_dir, exist_ok=True)
    snapshot_path = os.path.join(data_dir, f"twse_snapshot_{year}.parquet")

    train_start = pd.to_datetime(f"{year}-01-01") - pd.DateOffset(years=5)
    test_end = pd.to_datetime(f"{year}-12-31")

    new_snapshot = twse[(twse.index >= train_start) & (twse.index <= test_end)].copy()

    if os.path.exists(snapshot_path) and not overwrite:
        old_snapshot = pd.read_parquet(snapshot_path)
        if snapshot_hash(new_snapshot) == snapshot_hash(old_snapshot):
            print(f"快照未變動，載入快照：{snapshot_path}")
            return old_snapshot
    new_snapshot.to_parquet(snapshot_path)
    print(f"快照已更新：{snapshot_path}")
    return new_snapshot


def load_or_fail_model(model_path):
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"模型不存在：{model_path}")
    with open(model_path, 'rb') as f:
        model_dict = pickle.load(f)
    print(f"載入模型：{model_path}")
    return model_dict


def run_yearly_prediction(twse, start_year=2020, model_dir="saved_models_yearly", data_dir="snapshot_data"):
    results = []

    for year in range(start_year, twse.index[-1].year + 1):
        model_path = os.path.join(model_dir, f"ensemble_model_{year}.pkl")
        model_dict = load_or_fail_model(model_path)
        twse_snapshot = create_or_load_snapshot(twse, year, data_dir)

        # 生成 target
        twse_snapshot['target'] = (twse_snapshot['intraday_ret'].shift(-1) > 0).astype(float)

        # 特徵與標籤
        X = twse_snapshot[model_dict['features']].replace([np.inf, -np.inf], np.nan).fillna(0).clip(-1e6, 1e6)
        y = twse_snapshot['target']

        # 測試集篩選（僅當年）
        X_test = X[X.index.year == year]
        y_test = y.loc[X_test.index]

        if X_test.empty:
            print(f"{year} 沒有可用的測試資料，跳過。")
            continue

        scaler = model_dict['scaler']
        model = model_dict['model']

        # 預測
        X_test_scaled = scaler.transform(X_test)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        y_proba_0 = model.predict_proba(X_test_scaled)[:, 0]
        y_proba_diff = np.abs(y_proba - 0.5)
        y_proba_rank = pd.Series(y_proba).rank(method='first', pct=True).values  

        for idx, yt, yp, p1, p0, d, r in zip(X_test.index, y_test, y_pred, y_proba, y_proba_0, y_proba_diff, y_proba_rank):
            results.append({
                'date': idx,
                'y_true': yt if not np.isnan(yt) else None,
                'y_pred': yp,
                'y_proba': p1,
                'y_proba_0': p0,
                'y_proba_diff': d,
                'y_proba_rank': r
            })

        print(f"\n{year} 測試資料筆數: {len(X_test)}")
        # print(f"Scaler mean hash: {hash_array(scaler.mean_)}")
        # print(f"Scaler std hash : {hash_array(scaler.scale_)}")

    result_df = pd.DataFrame(results).set_index('date').sort_index()

    eval_df = result_df.dropna(subset=['y_true'])
    acc = accuracy_score(eval_df['y_true'], eval_df['y_pred'])
    baseline = max(eval_df['y_true'].mean(), 1 - eval_df['y_true'].mean())

    print("\n每年 retrain/載入 模型預測結果統計")
    print(f"Baseline Accuracy: {baseline:.4f}")
    print(f"Model Accuracy   : {acc:.4f}")

    return result_df


# === 回傳結果變數 ===
result_df = run_yearly_prediction(twse)
market_proba = pd.Series(result_df['y_proba'].values, index=result_df.index)
market_condition = pd.Series(result_df['y_pred'].values, index=result_df.index)

載入模型：saved_models_yearly\ensemble_model_2020.pkl
快照未變動，載入快照：snapshot_data\twse_snapshot_2020.parquet

2020 測試資料筆數: 245
載入模型：saved_models_yearly\ensemble_model_2021.pkl
快照未變動，載入快照：snapshot_data\twse_snapshot_2021.parquet

2021 測試資料筆數: 244
載入模型：saved_models_yearly\ensemble_model_2022.pkl
快照未變動，載入快照：snapshot_data\twse_snapshot_2022.parquet

2022 測試資料筆數: 246
載入模型：saved_models_yearly\ensemble_model_2023.pkl
快照未變動，載入快照：snapshot_data\twse_snapshot_2023.parquet

2023 測試資料筆數: 239
載入模型：saved_models_yearly\ensemble_model_2024.pkl
快照未變動，載入快照：snapshot_data\twse_snapshot_2024.parquet

2024 測試資料筆數: 242
載入模型：saved_models_yearly\ensemble_model_2025.pkl
快照已更新：snapshot_data\twse_snapshot_2025.parquet

2025 測試資料筆數: 67

每年 retrain/載入 模型預測結果統計
Baseline Accuracy: 0.5246
Model Accuracy   : 0.5814


In [36]:
dates = score.loc['2024-10-01':].index
short_weight_dict = {}

for date in tqdm.tqdm(dates):
    try:
        last_factor = factor[con].loc[date].drop(columns=刪除標的).dropna().sort_values(ascending=True)
        last_factor = last_factor[last_factor.index.isin(可借券名單)]
        singal_bool2 = last_factor.rank(ascending=True, method='first') <= low_n
        singal_list2 = singal_bool2[singal_bool2].index.to_list()
        score_today = score.loc[date].reindex(singal_list2).fillna(0)
        weights = score_today.abs() / score_today.abs().sum()
        short_weight_dict[date] = weights
    except Exception as e:
        print(f"{date} 發生錯誤：{e}")
        continue
short_weights = pd.DataFrame(short_weight_dict).T.fillna(0)

100%|██████████| 129/129 [00:04<00:00, 29.38it/s]


# 主要回測

In [37]:
# === 全域參數設定 ===
AUM門檻 = 50000000
low_n = 20

# === 回測主程式 ===
def run_backtest(factor, Handler, exp_ret, lend, 可借券名單):
    dates = Handler['Open'].loc['2024-10-01':].index
    報酬率_series = pd.Series(index=dates, dtype=float)
    資金使用率_series = pd.Series(index=dates, dtype=float)
    持倉數量_series = pd.Series(index=dates, dtype=int)
    提前回補紀錄 = []
    原始選股紀錄 = {}

    lend_qty = lend.set_index('coid')['Qty'] // 1000
    進場天數 = 0
    default_market_condition = 0  

    for i in tqdm.tqdm(range(1, len(dates))):
        if i >= len(dates):
            break

        最後一天 = dates[i - 1]
        當天 = dates[i]
        資金倍率 = 0 if market_condition.get(最後一天, default_market_condition) == 1 else 1
        if 資金倍率 == 0:
            continue
        進場天數 += 1

        try:
            當日因子 = factor[con].loc[最後一天].dropna().sort_values(ascending=True)
            當日因子 = 當日因子[當日因子.index.isin(可借券名單)]
            可借張數 = lend_qty.reindex(當日因子.index).fillna(0).astype(int)
            有張數 = 可借張數[可借張數 > 0].index
            當日因子 = 當日因子.loc[有張數]
            排序標的 = 當日因子.nsmallest(low_n).index
            原始選股紀錄[最後一天] = list(排序標的)
            
            short_signal = pd.DataFrame(0, index=[最後一天], columns=exp_ret.columns)
            short_signal.loc[最後一天, 排序標的] = -1

            score_weight = short_weights.loc[[最後一天]].reindex(columns=short_signal.columns).fillna(0)
            W = short_signal * score_weight
            W = W.div(abs(W.sum(axis=1)), axis=0)
            weighting = W.copy()

            股價 = Handler['Open'].loc[當天].reindex(weighting.columns)
            資金配置 = weighting.loc[最後一天] * AUM門檻 * 資金倍率
            預計張數 = (資金配置.abs() / (股價 * 1000)).fillna(0).astype(int)

            可借張數 = lend_qty.reindex(預計張數.index).fillna(0).astype(int)
            初步張數 = 預計張數.clip(upper=可借張數)
            已投入金額 = 初步張數 * 股價 * 1000
            剩餘資金 = AUM門檻 - 已投入金額.sum()

            最終張數 = 初步張數.copy()
            剩餘資金 = max(0, AUM門檻 - (最終張數 * 股價 * 1000).sum())

            可再加碼張數 = 可借張數 - 最終張數
            可再加碼張數 = 可再加碼張數[可再加碼張數 > 0]
            可再加碼標的 = 可再加碼張數.index.intersection(排序標的)

            if not 可再加碼標的.empty and 剩餘資金 > 0:
                vol = exp_ret[可再加碼標的].rolling(20).std().loc[最後一天].fillna(1e-6)
                low_vol_stocks = vol.nsmallest(len(vol)).index
                新增張數 = pd.Series(0, index=low_vol_stocks)

                for stock in low_vol_stocks:
                    股價_ = 股價.get(stock, np.nan)
                    if pd.isna(股價_):
                        continue
                    單張金額 = 股價_ * 1000
                    可加碼張數 = 可借張數[stock] - 最終張數[stock]
                    if 可加碼張數 <= 0:
                        continue
                    最多加碼張數 = min(可加碼張數, int(剩餘資金 // 單張金額))
                    if 最多加碼張數 <= 0:
                        continue
                    新增張數[stock] = 最多加碼張數
                    剩餘資金 -= 最多加碼張數 * 單張金額
                    if 剩餘資金 <= 0:
                        break

                if 新增張數.sum() == 0:
                    break
                最終張數.update(最終張數 + 新增張數)

            實際投入金額 = 最終張數 * 股價 * 1000 * np.sign(資金配置)
            adjusted_weight = 實際投入金額 / AUM門檻
            weighting.loc[最後一天] = adjusted_weight

            昨收 = Handler['Close'].loc[最後一天].reindex(最終張數.index)
            當日最低 = Handler['Low'].loc[當天].reindex(最終張數.index)
            當日收盤 = Handler['Close'].loc[當天].reindex(最終張數.index)
            跌停價 = (昨收 / 1.095).apply(lambda x: np.floor(x * 20) / 20 if pd.notna(x) else np.nan)

            補回價格 = 當日收盤.copy()
            跌停回補 = 當日最低 <= 跌停價
            補回價格[跌停回補] = 當日最低[跌停回補]

            提前回補股票 = 跌停回補[跌停回補].index
            weighting.loc[最後一天, 提前回補股票] = 0

            for stock in 提前回補股票:
                提前回補紀錄.append({'date': 當天, 'stock': stock})

            回補報酬 = (補回價格 - 股價) / 股價

            手續費_買 = 最終張數 * 股價 * 1000 * (0.001425 * 0.3 + 0.003 / 2)
            手續費_賣 = 最終張數 * 股價 * 1000 * (0.001425 * 0.3)
            總手續費 = (手續費_買 + 手續費_賣).sum()

            報酬金額 = (adjusted_weight * AUM門檻 * 回補報酬).sum()
            報酬率 = (報酬金額 - 總手續費) / AUM門檻 * 資金倍率

            報酬率_series.loc[最後一天] = 報酬率
            資金使用率_series.loc[最後一天] = weighting.loc[最後一天].abs().sum()
            持倉數量_series.loc[最後一天] = (weighting.loc[最後一天].abs() > 0).sum()

        except Exception as e:
            print(f"{最後一天} 發生錯誤：{e}")
            continue

    報酬率_series = 報酬率_series.dropna()
    資金使用率_series = 資金使用率_series.dropna()
    持倉數量_series = 持倉數量_series.dropna()
    累積報酬率 = 報酬率_series.cumsum()
    
    print(f"market_condition[{最後一天.date()}] = {market_condition.get(最後一天, default_market_condition)}")
    print(f"→ 當天是否進場（資金倍率）: {資金倍率}")
    print(f"今年策略總報酬率：{累積報酬率.iloc[-1]:.2%}")
    print(f"今年策略總報酬金額：{AUM門檻 * 累積報酬率.iloc[-1]:,.0f}")
    print(f"\n總共進場天數：{進場天數} 天")
    print(f"\n提前回補總次數: {len(提前回補紀錄)}")

    if 提前回補紀錄:
        提前_df = pd.DataFrame(提前回補紀錄)
        display(提前_df.groupby('date')['stock'].apply(list).to_frame(name='提前回補標的'))

    display(calculate_metrics(報酬率_series.to_frame(name='strategy')))
    累積報酬率.iplot(title='累積報酬率')
    報酬率_series.to_drawdown_series().iplot(title='報酬率回撤')
    資金使用率_series.iplot(title='每日資金使用率')
    持倉數量_series.iplot(title='每日持倉數量')

    return 報酬率_series, 資金使用率_series, 持倉數量_series, weighting, 原始選股紀錄


def get_daily_trade_info(date, factor, Handler, exp_ret, lend, 可借券名單):
    date = pd.to_datetime(date)
    lend_qty = lend.set_index('coid')['Qty'] // 1000

    try:
        date_idx = Handler['Open'].index.get_loc(date)
        隔日 = Handler['Open'].index[date_idx + 1]

        # 預設 market_condition = 偏空 => 可做空
        資金倍率 = 0 if market_condition.get(date, 0.5) == 1 else 1
        if 資金倍率 == 0:
            print(f"\n{date.date()}：預測大盤明日走高，當日未執行空單交易")
            return None

        # 條件過濾
        con = Handler['Value_Dollars'].rolling(20).mean().loc[date] > 25000000
        當日因子 = factor.loc[date][con].dropna()
        當日因子 = 當日因子[當日因子.index.isin(可借券名單)]
        可借張數 = lend_qty.reindex(當日因子.index).fillna(0).astype(int)
        有張數 = 可借張數[可借張數 > 0].index
        當日因子 = 當日因子.loc[有張數]
        排序標的 = 當日因子.nsmallest(low_n).index

        # 訊號與權重
        short_signal = pd.Series(0, index=factor.columns)
        short_signal.loc[排序標的] = -1
        score_weight = abs(當日因子.loc[排序標的])
        score_weight = score_weight / score_weight.sum()
        W = short_signal.loc[score_weight.index] * score_weight
        W = W / W.abs().sum()

        # 資金配置
        股價 = Handler['Open'].loc[隔日].reindex(W.index)
        資金配置 = W * AUM門檻 * 資金倍率
        預計張數 = (資金配置.abs() / (股價 * 1000)).fillna(0).astype(int)

        # 借券限制與初步建倉
        可借張數 = lend_qty.reindex(預計張數.index).fillna(0).astype(int)
        初步張數 = 預計張數.clip(upper=可借張數)
        已投入金額 = 初步張數 * 股價 * 1000
        剩餘資金 = AUM門檻 - 已投入金額.sum()

        最終張數 = 初步張數.copy()
        額外張數 = pd.Series(0, index=最終張數.index)

        # 加碼（低風險資金加碼）
        可再加碼張數 = 可借張數 - 最終張數
        可再加碼張數 = 可再加碼張數[可再加碼張數 > 0]
        可再加碼標的 = 可再加碼張數.index.intersection(排序標的)

        if not 可再加碼標的.empty and 剩餘資金 > 0:
            max_可再投入金額 = 可再加碼張數[可再加碼標的] * 股價[可再加碼標的] * 1000
            max_可再投入金額 = max_可再投入金額.fillna(0)

            total_可投 = max_可再投入金額.sum()
            if total_可投 > 0:
                比例分配金額 = (max_可再投入金額 / total_可投) * 剩餘資金
                新增張數 = (比例分配金額 / (股價[可再加碼標的] * 1000)).fillna(0).astype(int)
                新增張數 = 新增張數.clip(upper=可再加碼張數)
                最終張數.update(最終張數[可再加碼標的] + 新增張數)
                額外張數.update(新增張數)

        # 建倉資訊與報酬
        實際投入金額 = 最終張數 * 股價 * 1000 * np.sign(資金配置)
        adjusted_weight = 實際投入金額 / AUM門檻

        昨收 = Handler['Close'].loc[date].reindex(最終張數.index)
        當日最低 = Handler['Low'].loc[隔日].reindex(最終張數.index)
        當日收盤 = Handler['Close'].loc[隔日].reindex(最終張數.index)
        跌停價 = (昨收 / 1.095).apply(lambda x: np.floor(x * 20) / 20 if pd.notna(x) else np.nan)

        補回價格 = 當日收盤.copy()
        跌停回補 = 當日最低 <= 跌停價
        補回價格[跌停回補] = 當日最低[跌停回補]

        # 跌停回補股票歸零權重
        adjusted_weight.loc[跌停回補[跌停回補].index] = 0

        回補報酬 = (補回價格 - 股價) / 股價
        手續費_買 = 最終張數 * 股價 * 1000 * (0.001425 * 0.3 + 0.003 / 2)
        手續費_賣 = 最終張數 * 股價 * 1000 * (0.001425 * 0.3)
        總手續費 = 手續費_買 + 手續費_賣

        報酬金額 = adjusted_weight * AUM門檻 * 回補報酬
        淨報酬金額 = 報酬金額 - 總手續費
        報酬率 = 淨報酬金額.sum() / AUM門檻

        info_df = pd.DataFrame({
            '股價': 股價,
            '預計張數': 預計張數,
            '可借張數': 可借張數,
            '實際張數': 最終張數,
            '額外張數': 額外張數,
            '投入金額': 實際投入金額.abs(),
            '權重': (adjusted_weight.abs() * 100).round(4),
            '報酬率(%)': (回補報酬 * 100).round(4),
            '淨報酬金額': 淨報酬金額.round(0),
            '資金倍率': 資金倍率
        }).loc[排序標的].dropna()

        print(f"\n訊號日期: {date.date()}")
        print(f"交易檔數: {len(info_df)}")
        print(f"使用金額: {int(info_df['投入金額'].abs().sum())} 元")
        print(f"手續費: {int(總手續費.sum())} 元")
        print(f"資金使用率: {info_df['投入金額'].abs().sum() / AUM門檻:.2%}")
        print(f"報酬金額: {int(淨報酬金額.sum())} 元")
        print(f"當日報酬率: {報酬率:.2%}")

        return info_df

    except Exception as e:
        print(f"{date} 發生錯誤：{e}")
        return None


In [38]:
報酬率_series, 資金使用率_series,持倉_series, weighting, 原始選股紀錄 = run_backtest(factor[con], Handler, exp_ret,lend, 可借券名單)

100%|██████████| 128/128 [00:10<00:00, 12.25it/s]


market_condition[2025-04-17] = 1
→ 當天是否進場（資金倍率）: 0
今年策略總報酬率：48.06%
今年策略總報酬金額：24,027,923

總共進場天數：58 天

提前回補總次數: 2058


,提前回補標的
date,
2024-10-04,"[2603, 2609, 2613, 2615, 3430, 3580, 4113, 451..."
2024-10-09,"[1799, 2070, 2436, 2601, 3499, 3622, 5443, 610..."
2024-10-11,"[1294, 1721, 1786, 2062, 2069, 2070, 2425, 320..."
2024-10-15,"[1595, 3081, 3710, 5234, 6221, 6546]"
2024-10-24,"[1806, 2388, 3081, 3167, 3209, 3219, 3450, 523..."
2024-10-25,"[3029, 3115, 5205, 6144, 6546, 6763, 8272]"
2024-10-28,"[1307, 1528, 2399, 2477, 3040, 3081, 3234, 327..."
2024-11-05,"[1436, 1438, 1799, 2351, 3289, 5258, 5274, 614..."
2024-11-07,"[5205, 6114, 6873, 8464]"


,CAGR(%),Sharpe,Sortino,Calmar,MDD(%),樣本勝率(%),周勝率(%),月勝率(%),年勝率(%),預期報酬(bps),avg_win(%),avg_loss(%),avg_win/avg_loss,平均交易報酬(%),profit_factor
strategy,83.73,12.57,28.45,67.79,-1.24,77.59,75.86,100.0,100.0,82.32,1.17,-0.36,3.27,0.83,11.33


None

In [39]:
get_daily_trade_info('2025-4-10', factor, Handler, exp_ret, lend, 可借券名單)


2025-04-10：預測大盤明日走高，當日未執行空單交易


In [40]:
date = pd.to_datetime("2025-04-18")
low_n = 20
刪除標的 = []
指定交易标的 = []
有限購入張數 = {}

# === 選股邏輯 ===
last_factor = factor[con].loc[date].drop(columns=刪除標的).dropna().sort_values(ascending=True)
last_factor = last_factor[last_factor.index.isin(可借券名單)]
singal_bool2 = last_factor.rank(ascending=True, method='first') <= low_n
singal_list2 = singal_bool2[singal_bool2].index.to_list()

# === 價量資料 ===
Close = Handler['Close']
volume = Handler['Value_Dollars']
volume = (volume / 1e8).round(2)
收盤價2 = Close.drop(columns=刪除標的).loc[date][singal_list2]
成交量2 = volume.drop(columns=刪除標的)[singal_list2].rolling(20).mean().loc[date].round(2)

# === 報酬資料 ===
date_idx = Close.index.get_loc(date)
one_week_ago = Close.index[date_idx - 5]
一週前價 = Close.loc[one_week_ago, singal_list2]
今日價 = 收盤價2
近一週報酬率 = (今日價 / 一週前價 - 1).round(3) * 100

ind_ = ind[ind['mdate'] == date].set_index('coid')['產業別'].str.replace(r'^[A-Za-z0-9\s]+', '', regex=True)
產業別2 = ind_.reindex(singal_list2)
市場別2 = mkt[mkt['mdate'] == date].set_index('coid')['市場別'].reindex(singal_list2)
上市天數 = Close[singal_list2].notna().astype(int).sum()
上市天數.name = '上市上櫃天數'


是否可借券 = pd.Series(
    [coid in 可借券名單 for coid in singal_list2],
    index=singal_list2,
    name='是否可借券'
).map({True: 'Y', False: 'F'})
lend_qty = (lend.set_index('coid')['Qty'] // 1000).loc[singal_list2]  # 換算成張數（整數除法）

# === 訊號觸發條件 ===
market_prob = market_proba.get(date, 0.5)

# === 結果 ===
if market_prob > 0.5:
    print(f"預測大盤明日走高，不建議做空（機率 {market_prob:.2f}）")
elif market_prob < 0.5:
    print(f"預測大盤明日走低，建議做空（機率 {market_prob:.2f}）")
else:
    print(f"預測大盤漲跌機率均等（機率 {market_prob:.2f}")
預估權重 = short_weights.loc[date, singal_list2].fillna(0)

預估張數 = ((預估權重 * 50000000) / (收盤價2 * 1000)).fillna(0).astype(int)
訊號表 = pd.concat([
    收盤價2.rename('收盤價'),
    (收盤價2 / 1.095).rename('跌停價'),
    成交量2.rename('成交金額(億元)'),
    近一週報酬率.rename('近一週報酬率%'),
    產業別2.rename('產業別'),
    市場別2.rename('市場別'),
    上市天數,
    是否可借券.rename('是否可借券'),
    lend_qty.rename('可借券張數'),
    預估權重.rename('預估權重(%)').mul(100).round(2),
    預估張數.rename('預估張數')
], axis=1)

print(f'訊號日期: {date.date()}')
display(訊號表)

預測大盤明日走高，不建議做空（機率 0.66）
訊號日期: 2025-04-18


,收盤價,跌停價,成交金額(億元),近一週報酬率%,產業別,市場別,上市上櫃天數,是否可借券,可借券張數,預估權重(%),預估張數
1810,17.65,16.118721,1.58,5.4,玻璃陶瓷,TSE,3748,Y,8,2.17,61
1806,11.35,10.365297,0.58,7.1,玻璃陶瓷,TSE,3748,Y,32,5.76,253
6177,46.00,42.009132,1.66,5.9,建材營造,TSE,3748,Y,56,2.33,25
2634,47.30,43.196347,22.18,0.4,航運業,TSE,2598,Y,15,10.96,115
3546,92.00,84.018265,0.91,9.7,文化創意業,OTC,3748,Y,6,4.63,25
3162,42.75,39.041096,4.70,4.0,電機機械,OTC,3748,Y,7,2.59,30
8171,38.80,35.433790,0.78,10.5,綠能環保,OTC,3260,Y,6,4.14,53
4541,49.15,44.885845,22.21,15.4,其它,OTC,2621,Y,11,1.94,19
8042,31.00,28.310502,0.80,4.4,電子類,OTC,3748,Y,26,12.82,206
1316,15.10,13.789954,8.36,-5.3,建材營造,TSE,3748,Y,9,5.02,166
